In [1]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from ultralytics import YOLO, SAM 

In [2]:
from ultralytics import YOLO

if __name__ == '__main__':
    model = YOLO(r'E:\mastercode\6.yolo\ultralytics-main\ultralytics\cfg\models\11\yolo11-seg_DW.yaml')  # 可换 s/m/l/x
    model.train(
        data=r'E:\\mastercode\\6.yolo\\ultralytics-main\\voc20007_seg_yolov8.yaml',
        project=r'E:/mastercode/6.yolo/runs/segment',
        epochs=300,
        imgsz=640,
        batch=8,
        # lr0 = 0.00001,
        # lrf = 0.01,
        # momentum = 0.73,
        # weight_decay = 0.0005,
        # warmup_epochs = 3,
        # optimizer='AdamW',
        device=0  # CPU 则改为 'cpu'
    )

WARNING no model scale passed. Assuming scale='n'.


New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.19  Python-3.11.14 torch-2.5.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\\mastercode\\6.yolo\\ultralytics-main\\voc20007_seg_yolov8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=E:\mastercode\6.yol

In [3]:
YOLO_WEIGHT = r'E:\mastercode\6.yolo\runs\segment\train6\weights\best.pt'
SAM_WEIGHT  = r'E:\mastercode\6.yolo\sam2.1_t.pt'          # 自动下载，约375MB；也可用 sam_l.pt（更精细但更慢）

# 测试图片：单张路径 或 文件夹路径
INPUT      = r'E:/mastercode/data/VOC/yolo_voc/images/val'
OUTPUT_DIR = r'E:/mastercode/6.yolo/runs/yolo_sam_results'

CONF_THRESH = 0.25   # YOLO 置信度阈值
IOU_THRESH  = 0.45   # YOLO NMS IOU阈值
IMG_SIZE    = 640

VOC_CLASSES = [
    'aeroplane','bicycle','bird','boat','bottle',
    'bus','car','cat','chair','cow','diningtable',
    'dog','horse','motorbike','person','pottedplant',
    'sheep','sofa','train','tvmonitor'
]

# 每个类别一个颜色
np.random.seed(42)
CLASS_COLORS = np.random.randint(80, 230, size=(20, 3), dtype=np.uint8)


# ──────────────────────────────────────────────
# 核心推理函数
# ──────────────────────────────────────────────

def run_yolo_sam(yolo_model, sam_model, img_path, output_dir, device):
    img_path = Path(img_path)
    img_bgr  = cv2.imread(str(img_path))
    if img_bgr is None:
        print(f"  跳过：无法读取 {img_path}")
        return
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W    = img_rgb.shape[:2]

    # ── Step1：YOLO 推理 ──
    yolo_res = yolo_model(
        img_path,
        conf=CONF_THRESH,
        iou=IOU_THRESH,
        imgsz=IMG_SIZE,
        device=device,
        verbose=False
    )[0]

    if yolo_res.boxes is None or len(yolo_res.boxes) == 0:
        print(f"  {img_path.name}：YOLO 未检测到目标，跳过")
        return

    boxes  = yolo_res.boxes.xyxy.cpu().numpy()    # (N, 4)
    scores = yolo_res.boxes.conf.cpu().numpy()    # (N,)
    cls_ids= yolo_res.boxes.cls.cpu().numpy().astype(int)  # (N,)

    print(f"  {img_path.name}：检测到 {len(boxes)} 个目标")

    # ── Step2：SAM 精细化 ──
    sam_res   = sam_model(img_path, bboxes=boxes, verbose=False)[0]
    sam_masks = sam_res.masks  # ultralytics Masks 对象

    # ── Step3：可视化 ──
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(img_path.name, fontsize=12)

    # 左：原图
    axes[0].imshow(img_rgb)
    axes[0].set_title('原始图像')
    axes[0].axis('off')

    # 中：YOLO-seg 结果（如果有seg输出）
    yolo_vis = yolo_res.plot(labels=True, conf=True)
    yolo_vis = cv2.cvtColor(yolo_vis, cv2.COLOR_BGR2RGB)
    axes[1].imshow(yolo_vis)
    axes[1].set_title('YOLO-seg 分割')
    axes[1].axis('off')

    # 右：YOLO+SAM 精细掩码
    overlay = img_rgb.copy().astype(np.float32)
    legend_patches = []

    if sam_masks is not None and len(sam_masks) > 0:
        masks_data = sam_masks.data.cpu().numpy()  # (N, H, W)
        for i, (mask, cls_id, score) in enumerate(zip(masks_data, cls_ids, scores)):
            color = CLASS_COLORS[cls_id % 20].astype(np.float32)
            # 半透明填充
            mask_bool = mask.astype(bool)
            overlay[mask_bool] = overlay[mask_bool] * 0.4 + color * 0.6
            # 轮廓
            mask_uint8 = (mask * 255).astype(np.uint8)
            contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay.astype(np.uint8), contours, -1,
                             color.tolist(), 2)

            cls_name = VOC_CLASSES[cls_id] if cls_id < len(VOC_CLASSES) else str(cls_id)
            patch = mpatches.Patch(
                color=color / 255.0,
                label=f'{cls_name} {score:.2f}'
            )
            if cls_name not in [p.get_label().split()[0] for p in legend_patches]:
                legend_patches.append(patch)

    axes[2].imshow(overlay.astype(np.uint8))
    axes[2].set_title('YOLO + SAM 精细分割')
    axes[2].axis('off')
    if legend_patches:
        axes[2].legend(handles=legend_patches, loc='lower left',
                       fontsize=7, framealpha=0.7)

    plt.tight_layout()

    # 保存
    out_path = Path(output_dir) / (img_path.stem + '_yolo_sam.jpg')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  已保存 → {out_path}")


# ──────────────────────────────────────────────
# 主函数
# ──────────────────────────────────────────────

def main():
    device = '0' if torch.cuda.is_available() else 'cpu'
    print(f"使用设备: {'GPU' if device=='0' else 'CPU'}")

    # 加载模型
    print("加载 YOLO 模型...")
    yolo = YOLO(YOLO_WEIGHT)

    print("加载 SAM 模型（首次运行会自动下载）...")
    sam = SAM(SAM_WEIGHT)

    # 准备输出目录
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    # 收集图片
    input_path = Path(INPUT)
    if input_path.is_file():
        img_list = [input_path]
    else:
        img_list = list(input_path.glob('*.jpg')) + \
                   list(input_path.glob('*.jpeg')) + \
                   list(input_path.glob('*.png'))
        img_list = sorted(img_list)[:20]  # 先跑前20张看效果

    print(f"共 {len(img_list)} 张图片，开始推理...\n")

    for img_path in img_list:
        run_yolo_sam(yolo, sam, img_path, OUTPUT_DIR, device)

    print(f"\n全部完成！结果保存在：{OUTPUT_DIR}")


if __name__ == '__main__':
    main()


使用设备: GPU
加载 YOLO 模型...
加载 SAM 模型（首次运行会自动下载）...
共 20 张图片，开始推理...

  000039.jpg：检测到 1 个目标


C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 21407 (\N{CJK UNIFIED IDEOGRAPH-539F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 22987 (\N{CJK UNIFIED IDEOGRAPH-59CB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 20687 (\N{CJK UNIFIED IDEOGRAPH-50CF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 21

  已保存 → E:\mastercode\6.yolo\runs\yolo_sam_results\000039_yolo_sam.jpg
  000063.jpg：检测到 1 个目标


C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 21407 (\N{CJK UNIFIED IDEOGRAPH-539F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 22987 (\N{CJK UNIFIED IDEOGRAPH-59CB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 20687 (\N{CJK UNIFIED IDEOGRAPH-50CF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 21

  已保存 → E:\mastercode\6.yolo\runs\yolo_sam_results\000063_yolo_sam.jpg
  000121.jpg：检测到 3 个目标


...

  已保存 → E:\mastercode\6.yolo\runs\yolo_sam_results\000676_yolo_sam.jpg
  000713.jpg：YOLO 未检测到目标，跳过
  000720.jpg：检测到 2 个目标


C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 21407 (\N{CJK UNIFIED IDEOGRAPH-539F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 22987 (\N{CJK UNIFIED IDEOGRAPH-59CB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 20687 (\N{CJK UNIFIED IDEOGRAPH-50CF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 21

  已保存 → E:\mastercode\6.yolo\runs\yolo_sam_results\000720_yolo_sam.jpg
  000738.jpg：检测到 1 个目标


C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 21407 (\N{CJK UNIFIED IDEOGRAPH-539F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 22987 (\N{CJK UNIFIED IDEOGRAPH-59CB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 20687 (\N{CJK UNIFIED IDEOGRAPH-50CF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\33836\AppData\Local\Temp\ipykernel_15460\4217385244.py:110: UserWarning: Glyph 21

  已保存 → E:\mastercode\6.yolo\runs\yolo_sam_results\000738_yolo_sam.jpg

全部完成！结果保存在：E:/mastercode/6.yolo/runs/yolo_sam_results
